In [ ]:
# train_models.py
import re
import os
import json
import itertools
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import time
import torch.optim as optim
from utils import MLPTextGenerator, train_model

def load_and_process_text():
    file_list = [
        "Data/shake_sphere.txt",
        "Data/leo_tolstoys_war_and_peace.txt",
        "Data/paul_graham_essays.txt",
        "Data/the_adventure_of_sherlock_holmes.txt"
    ]
    
    print("Loading and processing text data (this will run only once)...")
    all_words = []
    for file_path in file_list:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        all_words.extend(['.'])
                        continue
                    line = line.replace('.', ' . '); line = re.sub('[^a-zA-Z0. ]', '', line); line = line.lower()
                    all_words.extend(line.split())
        except FileNotFoundError:
            print(f"Warning: File not found {file_path}. Skipping.")
            
    all_words.append('.')
    print(f"Total words in corpus: {len(all_words)}")

    vocab = sorted(list(set(all_words)))
    word_to_ix = {word: i for i, word in enumerate(vocab)}
    ix_to_word = {i: word for i, word in enumerate(vocab)}

    word_counts = Counter(all_words)
    print(f"Total Vocabulary Size: {len(vocab)}\n")
    print("10 Most Frequent Words: - ", word_counts.most_common(10))
    print("10 Least Frequent Words: - ", word_counts.most_common()[:-11:-1])
    
    return all_words, word_to_ix, ix_to_word

def create_training_data(all_words, word_to_ix, CONTEXT_SIZE, TEST_SPLIT, BATCH_SIZE):
    """
    Generates X/y pairs and DataLoaders for a *specific* CONTEXT_SIZE.
    """
    print(f"Creating training data for CONTEXT_SIZE={CONTEXT_SIZE}...")
    X_indices, y_indices = [], []
    for i in range(len(all_words) - CONTEXT_SIZE):
        context = all_words[i : i + CONTEXT_SIZE]
        target = all_words[i + CONTEXT_SIZE]
        X_indices.append([word_to_ix[w] for w in context])
        y_indices.append(word_to_ix[target])
        
    print(f"Data pairs created. X shape: ({len(X_indices)}, {len(X_indices[0])}), y shape: ({len(y_indices)})")

    # Convert to tensors
    X_tensor = torch.tensor(X_indices, dtype=torch.long)
    y_tensor = torch.tensor(y_indices, dtype=torch.long)
    
    # Create dataset
    dataset = TensorDataset(X_tensor, y_tensor)
    val_size = int(len(dataset) * TEST_SPLIT)
    train_size = len(dataset) - val_size
    
    # Use sequential split (as in your original code)
    train_dataset = TensorDataset(X_tensor[:train_size], y_tensor[:train_size])
    val_dataset = TensorDataset(X_tensor[train_size:], y_tensor[train_size:])
    
    # Create DataLoaders
    # Using num_workers=2 and pin_memory=True as in your original
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    print(f"Created data loaders. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
    return train_loader, val_loader

if __name__ == "__main__":
    HIDDEN_DIM = 1024
    LEARNING_RATE = 0.001
    EPOCHS = 50
    BATCH_SIZE = 1024
    TEST_SPLIT = 0.1
    SEED = 43

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Global device set to: {device}")

    # --- Define Model Variant *Options* ---
    context_sizes = [5, 10, 15]
    embedding_dims = [32, 64]
    num_hidden_layers_list = [1, 2, 3]
    activations = ['relu', 'tanh']

    # --- Generate All Combinations ---
    print("Generating all model configurations...")
    model_configs = []
    
    # Use itertools.product to get the Cartesian product
    all_combinations = itertools.product(
        context_sizes, 
        embedding_dims, 
        num_hidden_layers_list, 
        activations
    )

    for (cs, ed, hl, act) in all_combinations:
        # Create a descriptive name for the file
        act_name = "Relu" if act == "relu" else "Tanh"
        name = f"CS{cs}_ED{ed}_HL{hl}_{act_name}"
        
        config = {
            "name": name,
            "CONTEXT_SIZE": cs,
            "EMBEDDING_DIM": ed,
            "NUM_HIDDEN_LAYERS": hl,
            "ACTIVATION": act,
        }
        model_configs.append(config)

    print(f"Generated {len(model_configs)} model configurations to train.")

    # --- Setup ---
    output_dir = "Models"
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Load data and build vocab ONCE
    all_words, word_to_ix, ix_to_word = load_and_process_text()
    VOCAB_SIZE = len(word_to_ix)
    
    # 2. Save the vocabulary ONCE
    vocab_path = os.path.join(output_dir, "vocab.json")
    with open(vocab_path, 'w') as f:
        json.dump({'word_to_ix': word_to_ix, 'ix_to_word': ix_to_word}, f)
    print(f"Vocabulary saved to {vocab_path}\n")

    # 3. Loop, Train, and Save each model variant
    total_configs = len(model_configs)
    for i, config in enumerate(model_configs):
        print(f"\n--- Training Model {i+1}/{total_configs}: {config['name']} ---")
        
        # A. Create data loaders for this specific config's CONTEXT_SIZE
        train_loader, val_loader = create_training_data(
            all_words, 
            word_to_ix, 
            config['CONTEXT_SIZE'], 
            TEST_SPLIT, 
            BATCH_SIZE
        )
        
        # B. Define model hyperparameters
        model_params = {
            'VOCAB_SIZE': VOCAB_SIZE,
            'EMBEDDING_DIM': config['EMBEDDING_DIM'],
            'CONTEXT_SIZE': config['CONTEXT_SIZE'],
            'NUM_HIDDEN_LAYERS': config['NUM_HIDDEN_LAYERS'],
            'HIDDEN_LAYER_DIM': HIDDEN_DIM,
            'ACTIVATION_TYPE': config['ACTIVATION'],
        }

        # C. Initialize Model
        model = MLPTextGenerator(**model_params)
        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
            model = nn.DataParallel(model)
        
        # D. Train Model
        history, trained_model = train_model(
            model, 
            train_loader, 
            val_loader, 
            LEARNING_RATE, 
            EPOCHS, 
            device
        )
        
        # E. Save the model
        save_path = os.path.join(output_dir, f"{config['name']}.pth")
        
        # Save a checkpoint dictionary
        torch.save({
            'hyperparameters': model_params,
            'model_state_dict': trained_model.state_dict(),
        }, save_path)
        
        print(f"Model {i+1}/{total_configs} ('{config['name']}') saved to {save_path}")
        print("-" * (30 + len(config['name'])) + "\n")

    print(f"--- ALL {total_configs} MODELS TRAINED AND SAVED ---")

Global device set to: cuda
Generating all model configurations...
Generated 36 model configurations to train.
Loading and processing text data (this will run only once)...
Total words in corpus: 116587
Total Vocabulary Size: 8590

10 Most Frequent Words: -  [('.', 9107), ('the', 5811), ('and', 3066), ('i', 2995), ('of', 2779), ('to', 2763), ('a', 2685), ('in', 1818), ('that', 1750), ('it', 1710)]
10 Least Frequent Words: -  [('newsletter', 1), ('subscribe', 1), ('includes', 1), ('pg', 1), ('edition', 1), ('necessarily', 1), ('network', 1), ('originator', 1), ('hart', 1), ('michael', 1)]
Vocabulary saved to Models\vocab.json


--- Training Model 1/36: CS5_ED32_HL1_Relu ---
Creating training data for CONTEXT_SIZE=5...
Data pairs created. X shape: (116582, 5), y shape: (116582)
Created data loaders. Train batches: 103, Val batches: 12
Starting training on cuda for up to 50 epochs...
  [Epoch 1] Batch 103/103 processed. (0 left)
Epoch 1/50 | Time: 7.72s | Train Loss: 6.2439 | Val Loss: 6.4

KeyboardInterrupt: 